# Anomaly detection with gearbox vibration

Anomaly detection asks whether a new observation differs from normal operation.
This example treats healthy GFD vibration windows as the reference distribution
and uses broken-tooth recordings only for evaluation.

## Learning goals

- build window-level vibration features with Polars;
- avoid training an anomaly detector on known faults;
- evaluate anomaly scores against held-out healthy and faulty windows.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdmdata").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import pdmdata
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

In [ ]:
pdmdata.download("gfd")

def vibration_windows(condition, load, window_size=256):
    frame = pdmdata.load("gfd", condition=condition, load=load)
    sensors = [column for column in frame.columns if column.startswith("sensor_")]
    return (
        frame.with_row_index("sample")
        .with_columns((pl.col("sample") // window_size).alias("window"))
        .group_by("window")
        .agg(
            *[pl.col(column).mean().alias(f"{column}_mean") for column in sensors],
            *[pl.col(column).std().alias(f"{column}_std") for column in sensors],
            *[
                (pl.col(column) ** 2).mean().sqrt().alias(f"{column}_rms")
                for column in sensors
            ],
        )
        .with_columns(
            pl.lit(condition).alias("condition"),
            pl.lit(load).alias("load"),
        )
    )

healthy = pl.concat([vibration_windows("healthy", load) for load in range(0, 100, 10)])
broken = pl.concat([vibration_windows("broken", load) for load in range(0, 100, 10)])
healthy.shape, broken.shape

## Train on normal operation

The split is made by load so that evaluation is not performed on the same
operating conditions used to fit the detector.

In [ ]:
feature_columns = [
    column for column in healthy.columns
    if column not in {"window", "condition", "load"}
]
train = healthy.filter(pl.col("load") < 70)
test = pl.concat([
    healthy.filter(pl.col("load") >= 70),
    broken.filter(pl.col("load") >= 70),
])

detector = IsolationForest(contamination="auto", random_state=0)
detector.fit(train.select(feature_columns).to_numpy())
anomaly_score = -detector.score_samples(test.select(feature_columns).to_numpy())
is_fault = (test.get_column("condition") == "broken").cast(pl.UInt8).to_numpy()
print(f"Window-level ROC AUC: {roc_auc_score(is_fault, anomaly_score):.3f}")

In [ ]:
scored = test.select("condition", "load").with_columns(
    pl.Series("anomaly_score", anomaly_score)
)
for condition, color in [("healthy", "tab:blue"), ("broken", "tab:red")]:
    values = scored.filter(pl.col("condition") == condition)["anomaly_score"].to_numpy()
    plt.hist(values, bins=30, alpha=0.55, label=condition, color=color)
plt.xlabel("Isolation Forest anomaly score")
plt.ylabel("Windows")
plt.legend()
plt.show()

## Interpretation and limitations

A high score means dissimilarity from the healthy training windows, not a
diagnosed failure. Load is a confounder, so a serious experiment should compare
condition-aware normalization, multiple window sizes, and group-based
cross-validation.